# W9D4 — CrewAI + LangChain: Hybrid Agent Systems

## Learning Objectives
1. Build a LangChain chain using PromptTemplate, Ollama LLM, and an output parser.
2. Test the chain with five inputs.
3. Add conversation memory and verify history across five turns.
4. Build an agent with web-search and calculator tools.
5. Run three agent tasks and save evidence.

In [1]:
import sys
from importlib.metadata import version, PackageNotFoundError

print("Python:", sys.executable)

for package in [
    "langchain",
    "langchain-core",
    "langchain-community",
    "langchain-ollama",
    "crewai",
]:
    try:
        print(f"{package}: {version(package)}")
    except PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

Python: c:\Users\ughan\CynarisInternship\AIML_PROJECT\.venv-w9d1\Scripts\python.exe
langchain: NOT INSTALLED
langchain-core: NOT INSTALLED
langchain-community: NOT INSTALLED
langchain-ollama: NOT INSTALLED
crewai: 1.15.28


In [2]:
import langchain
import langchain_core
import langchain_community
import langchain_ollama

print("LangChain:", langchain.__version__)
print("LangChain Core:", langchain_core.__version__)
print("LangChain Community:", langchain_community.__version__)
print("LangChain Ollama: IMPORT OK")

from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_ollama import OllamaLLM

print("PromptTemplate: OK")
print("StrOutputParser: OK")
print("OllamaLLM: OK")

C:\Users\ughan\AppData\Local\Temp\ipykernel_20032\3664741254.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  import langchain_community


LangChain: 1.4.7
LangChain Core: 1.6.13
LangChain Community: 0.4.2
LangChain Ollama: IMPORT OK
PromptTemplate: OK
StrOutputParser: OK
OllamaLLM: OK


In [3]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_ollama import OllamaLLM

prompt = PromptTemplate.from_template(
    "You are a concise robotics assistant. "
    "Answer the following question in 2-3 sentences:\n{question}"
)

llm = OllamaLLM(
    model="qwen2.5:3b",
    base_url="http://localhost:11434",
    temperature=0
)

output_parser = StrOutputParser()

chain = prompt | llm | output_parser

result = chain.invoke({
    "question": "Give two applications of computer vision in robotics."
})

print("Question: Give two applications of computer vision in robotics.")
print("\nAnswer:\n", result)

Question: Give two applications of computer vision in robotics.

Answer:
 Computer vision is applied in robotics for object recognition and tracking, enabling robots to identify and manipulate objects in their environment. Another application is in navigation, where computer vision helps robots understand their surroundings and navigate through complex spaces without human intervention.


In [4]:
test_questions = [
    "What is computer vision in robotics?",
    "Give two applications of computer vision in robotics.",
    "What is the role of sensors in autonomous robots?",
    "Explain obstacle avoidance in one or two sentences.",
    "How can robots use AI in smart agriculture?"
]

results = []

for i, question in enumerate(test_questions, start=1):
    answer = chain.invoke({"question": question})
    results.append({
        "test": i,
        "question": question,
        "answer": answer
    })
    print(f"\n--- Test {i}/5 ---")
    print("Question:", question)
    print("Answer:", answer)

print(f"\nCompleted {len(results)}/{len(test_questions)} tests.")


--- Test 1/5 ---
Question: What is computer vision in robotics?
Answer: Computer vision in robotics refers to the ability of robots to interpret and understand the visual world through sensors and algorithms, enabling them to perceive their environment, recognize objects, and interact with it autonomously.

--- Test 2/5 ---
Question: Give two applications of computer vision in robotics.
Answer: Computer vision is applied in robotics for object recognition and tracking, enabling robots to identify and manipulate objects in their environment. Another application is in navigation, where computer vision helps robots understand their surroundings and navigate through complex spaces without human intervention.

--- Test 3/5 ---
Question: What is the role of sensors in autonomous robots?
Answer: Sensors in autonomous robots play a crucial role by enabling the robots to perceive their environment, detect changes, and gather data, which they use to make decisions and perform tasks independentl

In [5]:
try:
    from langchain_classic.memory import ConversationBufferMemory
    print("ConversationBufferMemory: IMPORT OK")
except ImportError as e:
    print("Import failed:", e)

ConversationBufferMemory: IMPORT OK


In [6]:
from langchain_classic.memory import ConversationBufferMemory
from langchain_core.prompts import PromptTemplate
from langchain_ollama import OllamaLLM

memory = ConversationBufferMemory(
    memory_key="chat_history",
    input_key="input",
    output_key="response",
    return_messages=False
)

memory_prompt = PromptTemplate.from_template(
    """You are a helpful robotics assistant.
Conversation so far:
{chat_history}

User: {input}
Assistant: Answer briefly and use the conversation history when relevant."""
)

memory_llm = OllamaLLM(
    model="qwen2.5:3b",
    base_url="http://localhost:11434",
    temperature=0
)

def memory_turn(user_input):
    history = memory.load_memory_variables({})["chat_history"]
    response = (memory_prompt | memory_llm).invoke({
        "chat_history": history,
        "input": user_input
    })
    memory.save_context(
        {"input": user_input},
        {"response": response}
    )
    return response

turns = [
    "My robot's name is Atlas.",
    "Atlas is used for agricultural work.",
    "What is my robot's name?",
    "What type of work does my robot do?",
    "Suggest one AI feature for my robot."
]

for i, user_input in enumerate(turns, start=1):
    response = memory_turn(user_input)
    print(f"\n--- Turn {i}/5 ---")
    print("User:", user_input)
    print("Assistant:", response)

print("\nTotal conversation turns saved:", len(memory.chat_memory.messages) // 2)

C:\Users\ughan\AppData\Local\Temp\ipykernel_20032\2064748325.py:5: LangChainDeprecationWarning: The class `ConversationBufferMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationBufferMemory(
c:\Users\ughan\CynarisInternship\AIML_PROJECT\.venv-w9d1\Lib\site-packages\pydantic\main.py:250: LangChainDeprecationWarning: The class `InMemoryChatMessageHistory` was deprecated in LangChain 1.6.4 and will be removed in 2.0.0 See the short-term memory documentation for recommended alternatives: https://docs.langchain.com/oss/python/langchain/short-term-memory
  validated_self = self.__pydantic_validator__.validate_python(data, self_instance=self)



--- Turn 1/5 ---
User: My robot's name is Atlas.
Assistant: That's a great name for your robot! Atlas is known for strength and support. How can I assist you with Atlas today?

--- Turn 2/5 ---
User: Atlas is used for agricultural work.
Assistant: Great! Atlas can be very useful in agricultural applications, such as crop monitoring, soil analysis, and even assisting with harvesting. How specifically would you like to use Atlas for agricultural work?

--- Turn 3/5 ---
User: What is my robot's name?
Assistant: Your robot's name is Atlas. How can I assist you with Atlas today?

--- Turn 4/5 ---
User: What type of work does my robot do?
Assistant: Your robot, Atlas, is used for agricultural work. How specifically would you like to use Atlas for agricultural tasks?

--- Turn 5/5 ---
User: Suggest one AI feature for my robot.
Assistant: For your agricultural robot Atlas, one AI feature you might consider implementing is **remote monitoring and control**. This allows you to operate and monit

In [7]:
from importlib.metadata import version, PackageNotFoundError

for package in ["duckduckgo-search", "ddgs", "numexpr", "crewai"]:
    try:
        print(f"{package}: {version(package)}")
    except PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

duckduckgo-search: NOT INSTALLED
ddgs: 9.16.0
numexpr: NOT INSTALLED
crewai: 1.15.28


In [8]:
import ast
import operator
from ddgs import DDGS
from langchain_core.tools import tool

@tool
def web_search(query: str) -> str:
    """Search the web for recent information about a topic."""
    try:
        results = DDGS().text(query, max_results=3)
        if not results:
            return "No search results found."
        return "\n".join(
            f"{i}. {item.get('title', 'No title')}: "
            f"{item.get('body', 'No summary')}"
            for i, item in enumerate(results, start=1)
        )
    except Exception as e:
        return f"Web search error: {e}"

_ALLOWED_OPERATORS = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.Pow: operator.pow,
    ast.USub: operator.neg,
    ast.UAdd: operator.pos,
}

def _safe_calculate(node):
    if isinstance(node, ast.Constant) and type(node.value) in (int, float):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _ALLOWED_OPERATORS:
        left = _safe_calculate(node.left)
        right = _safe_calculate(node.right)
        if isinstance(node.op, ast.Pow) and abs(right) > 10:
            raise ValueError("Exponent is too large.")
        return _ALLOWED_OPERATORS[type(node.op)](left, right)
    if isinstance(node, ast.UnaryOp) and type(node.op) in _ALLOWED_OPERATORS:
        return _ALLOWED_OPERATORS[type(node.op)](_safe_calculate(node.operand))
    raise ValueError("Only basic arithmetic expressions are allowed.")

@tool
def calculator(expression: str) -> str:
    """Calculate a basic arithmetic expression, such as (25 * 4) + 10."""
    try:
        tree = ast.parse(expression, mode="eval")
        result = _safe_calculate(tree.body)
        return str(result)
    except Exception as e:
        return f"Calculation error: {e}"

print("Tools created:", web_search.name, "and", calculator.name)
print("\nCalculator test:", calculator.invoke({"expression": "(25 * 4) + 10"}))
print("\nWeb search test:")
print(web_search.invoke({"query": "ROS 2 robotics framework"}))

Tools created: web_search and calculator

Calculator test: 110

Web search test:
1. Robot Operating System - Wikipedia: Robot Operating System (ROS or ros) is an open-source robotics middleware suite. Although ROS is not an operating system (OS) but a set of software frameworks for robot software development, it provides services designed for a heterogeneous computer cluster such as hardware abstraction, low-level device control, implementation of commonly used functionality, message-passing between processes ...
2. ROS - Robot Operating System - ROS: Home: ROS - Robot Operating System The Robot Operating System (ROS) is a set of software libraries and tools that help you build robot applications. From drivers to state-of-the-art algorithms, and with powerful developer tools, ROS has what you need for your next robotics project. And it's all open source.
3. ROS 2 Documentation - Robot Operating System: The Robot Operating System (ROS) is a set of software libraries and tools for buildi

In [9]:
from langchain.agents import create_agent
from langchain_ollama import ChatOllama

agent_model = ChatOllama(
    model="qwen2.5:3b",
    base_url="http://localhost:11434",
    temperature=0
)

agent = create_agent(
    model=agent_model,
    tools=[web_search, calculator],
    system_prompt=(
        "You are a robotics research assistant. "
        "Use the calculator for arithmetic and web_search for factual web queries. "
        "Give concise answers based on tool results."
    )
)

agent_tasks = [
    "Calculate the total cost of 8 sensors at ₹450 each, plus ₹1200 for a controller.",
    "Search the web for the official ROS 2 documentation and summarize what ROS 2 provides.",
    "A robot travels 150 metres in 30 seconds. Use the calculator to find its average speed in metres per second."
]

for i, task in enumerate(agent_tasks, start=1):
    print(f"\n{'=' * 15} AGENT TASK {i}/3 {'=' * 15}")
    print("Task:", task)
    try:
        result = agent.invoke({
            "messages": [{"role": "user", "content": task}]
        })
        final_message = result["messages"][-1]
        print("Answer:", final_message.content)
        print("Messages in agent trace:", len(result["messages"]))
    except Exception as e:
        print("Agent task failed:", repr(e))


=============== AGENT TASK 1/3 ===============
Task: Calculate the total cost of 8 sensors at ₹450 each, plus ₹1200 for a controller.
Answer: The total cost for 8 sensors at ₹450 each, plus ₹1200 for a controller is ₹4800.
Messages in agent trace: 4

=============== AGENT TASK 2/3 ===============
Task: Search the web for the official ROS 2 documentation and summarize what ROS 2 provides.
Answer: The official ROS 2 documentation provides best practices and methodology for contributing code, documentation, and other improvements to ROS 2. It also includes instructions for migrating existing ROS 1 documentation to ROS 2. Additionally, the site contains documentation for specific ROS packages like MoveIt, image_proc, and octomap. For general ROS 1 documentation, one can refer to the ROS wiki.
Messages in agent trace: 4

=============== AGENT TASK 3/3 ===============
Task: A robot travels 150 metres in 30 seconds. Use the calculator to find its average speed in metres per second.
Answer: T

In [10]:
from pathlib import Path
import json

output_dir = Path("../outputs/W9D4")
output_dir.mkdir(parents=True, exist_ok=True)

evidence = {
    "day": "W9D4",
    "chain_tests": {
        "passed": 5,
        "total": 5
    },
    "memory_tests": {
        "passed": 5,
        "total": 5,
        "recall_verified": True
    },
    "tools": {
        "web_search": "passed",
        "calculator": "passed",
        "calculator_test": "(25 * 4) + 10 = 110"
    },
    "agent_tasks": [
        {
            "task": "Calculate sensor and controller cost",
            "expected": 4800,
            "observed": 4800,
            "passed": True
        },
        {
            "task": "Search and summarize ROS 2 documentation",
            "observed": "Summary returned; source verification remains limited",
            "passed": True
        },
        {
            "task": "Calculate average robot speed",
            "expected_m_per_s": 5,
            "observed_m_per_s": 5,
            "passed": True
        }
    ],
    "notes": [
        "ConversationBufferMemory and InMemoryChatMessageHistory emitted deprecation warnings.",
        "The web-search summary was not independently checked against the official documentation."
    ]
}

evidence_path = output_dir / "evaluation_results.json"
evidence_path.write_text(
    json.dumps(evidence, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("Evidence saved to:", evidence_path.resolve())
print("Agent tasks passed:", sum(t["passed"] for t in evidence["agent_tasks"]), "/ 3")

Evidence saved to: C:\Users\ughan\CynarisInternship\AIML_PROJECT\outputs\W9D4\evaluation_results.json
Agent tasks passed: 3 / 3


In [11]:
from pathlib import Path

output_dir = Path("../outputs/W9D4")
output_dir.mkdir(parents=True, exist_ok=True)

validation_report = """# W9D4 — Validation Report

## 1. LangChain Chain
- Status: PASS
- Pipeline: PromptTemplate → OllamaLLM → StrOutputParser
- Tests completed: 5/5

## 2. Conversation Memory
- Status: PASS
- Implementation: ConversationBufferMemory
- Turns tested: 5
- Recall verified: Robot name (Atlas) and purpose (agricultural work)
- Note: Deprecation warnings were emitted by legacy memory components.

## 3. Tools
- Web search: PASS — retrieved results about ROS 2.
- Calculator: PASS — (25 * 4) + 10 = 110.

## 4. Agent Tasks
1. Cost calculation: PASS — 8 × ₹450 + ₹1200 = ₹4800.
2. ROS documentation search: PASS — summary returned; exact source claims were not independently verified.
3. Robot average speed: PASS — 150 m / 30 s = 5 m/s.

## 5. Evidence
- `evaluation_results.json`

## 6. Limitations
- ConversationBufferMemory and InMemoryChatMessageHistory emitted deprecation warnings.
- The web-search agent's summary needs independent checking against the official ROS 2 documentation.
- Tests were run using the local Ollama model `qwen2.5:3b`.

## Overall Result
Core chain, five-turn memory, both tools, and three agent tasks were exercised successfully.
"""

report_path = output_dir / "VALIDATION.md"
report_path.write_text(validation_report, encoding="utf-8")

print("Validation report saved to:", report_path.resolve())
print("Report exists:", report_path.exists())

Validation report saved to: C:\Users\ughan\CynarisInternship\AIML_PROJECT\outputs\W9D4\VALIDATION.md
Report exists: True
